In [1]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

file_name = "yelp_academic_dataset_business.json"
full_path = f"{path}/{file_name}"

df_business = pd.read_json(full_path, lines=True)

print(df_business.head())

              business_id                      name  \
0  Pns2l4eNsfO8kk83dixA6A  Abby Rappoport, LAC, CMQ   
1  mpf3x-BjTdTEA3yCZrAYPw             The UPS Store   
2  tUFrWirKiKi_TAnsVWINQQ                    Target   
3  MTSW4McQd7CbVtyjqoe9mw        St Honore Pastries   
4  mWMc6_wTdE0EUBKIGXDVfA  Perkiomen Valley Brewery   

                           address           city state postal_code  \
0           1616 Chapala St, Ste 2  Santa Barbara    CA       93101   
1  87 Grasso Plaza Shopping Center         Affton    MO       63123   
2             5255 E Broadway Blvd         Tucson    AZ       85711   
3                      935 Race St   Philadelphia    PA       19107   
4                    101 Walnut St     Green Lane    PA       18054   

    latitude   longitude  stars  review_count  is_open  \
0  34.426679 -119.711197    5.0             7        0   
1  38.551126  -90.335695    3.0            15        1   
2  32.223236 -110.880452    3.5            22        0   
3  39.9555

In [2]:


# Keywords ki list jo food delivery aur dining ko cover karti hai
food_keywords = 'Restaurant|Food|Coffee|Tea|Cafe|Bakery|Dessert|Delivery|Takeout|Bars'

# Filter apply karna
df_food_businesses = df_business[
    df_business['categories'].fillna('').str.contains(food_keywords, case=False, na=False)
].copy()


In [3]:
import kagglehub
import pandas as pd

# Redefine path in this cell to ensure it's available
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

# 2. Specify the file you want to load (e.g., review or Review)
# Note: The Yelp files are "JSON Lines" format (one JSON object per line)
file_name = "yelp_academic_dataset_review.json"
full_path = f"{path}/{file_name}"

full_path
print('executed')

executed


In [4]:
import duckdb

# full_path is already defined and points to yelp_academic_dataset_review.json

# SQL query to load the review data directly from the JSON file using duckdb's read_json_auto
# and then join with df_food_businesses. Limiting to 100,000 rows as suggested for review data.
query = f"""
SELECT
    r.*
FROM
    read_json_auto('{full_path}', records='true') AS r
JOIN
    df_food_businesses AS b
ON
    r.business_id = b.business_id

"""
filtered_df = duckdb.sql(query).df()
filtered_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,rW7EKK_Iy14to6jd5tQQXw,aqa6jXdKEOQp9blzIIMleA,jQBPO3rYkNwIaOdQS5ktgQ,5.0,0,0,0,This little unassuming gem is one of the best ...,2016-03-20 14:03:59
1,RUBGABqzQdFJalHXRaIkWQ,e4KFQx7NvNZx1wPuYgfL5g,I0I2mkCOPTYVSLpc8A8VVw,5.0,0,0,0,Zack the bartender gets 10 stars. Made us phen...,2016-04-12 20:47:13
2,-aWiLFK1v5MBM6WZ88sMjQ,yqEeCDu5YEw3MI6HtsAT_w,GBTPC53ZrG1ZBY3DT8Mbcw,4.0,0,0,0,"Had the schnitzel, shrimp & grits, oysters, an...",2015-05-28 00:09:31
3,v_7qZ1jEjGvQnw3iXzEHrg,QmkMmFpxBytWiP4dRS91kw,ORL4JE6tz3rJxVqkdKfegA,5.0,0,0,0,Amazing. This hotel is one of the best I've ev...,2018-08-14 20:51:50
4,Iy_GyYyosqdY9gjlbKrPNA,uoDRDI_9GL8g9qg9msfHKw,isvL0ni23esix-j94Gw5wQ,5.0,2,0,0,"This is and unpretentious, family owned busine...",2017-09-01 22:21:29


In [5]:
import kagglehub
import pandas as pd

# Redefine path in this cell to ensure it's available
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

# 2. Specify the file you want to load (e.g., review or Review)
# Note: The Yelp files are "JSON Lines" format (one JSON object per line)
file_name = "yelp_academic_dataset_checkin.json"
full_path = f"{path}/{file_name}"

full_path

# query = f"""
# SELECT
#     r.*
# FROM
#     read_json_auto('{full_path}', records='true') AS r
# JOIN
#     df_food_businesses AS b
# ON
#     r.business_id = b.business_id

# """
# df_chekin = duckdb.sql(query).df()


df_chekin = pd.read_json(full_path, lines=True)
df_chekin.head()

,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22"
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012..."
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014..."


In [6]:
df_combined = pd.merge(df_food_businesses, filtered_df, on='business_id', how='inner')
df_combined = pd.merge(df_combined, df_chekin, on='business_id', how='left')
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5243722 entries, 0 to 5243721
Data columns (total 23 columns):
 #   Column        Dtype         
---  ------        -----         
 0   business_id   object        
 1   name          object        
 2   address       object        
 3   city          object        
 4   state         object        
 5   postal_code   object        
 6   latitude      float64       
 7   longitude     float64       
 8   stars_x       float64       
 9   review_count  int64         
 10  is_open       int64         
 11  attributes    object        
 12  categories    object        
 13  hours         object        
 14  review_id     object        
 15  user_id       object        
 16  stars_y       float64       
 17  useful        int64         
 18  funny         int64         
 19  cool          int64         
 20  text          object        
 21  date_x        datetime64[us]
 22  date_y        object        
dtypes: datetime64[us](1), float64(4)

## Objective:

## Q1.a — Cuisine-Level Performance & Consistency Analysis
Using business and review tables from the yelp dataset to answer following questions:
For each cuisine category (parsed from the categories field) extract the following information
from the above tables:
1. Total restaurants
2. Total reviews
3. Weighted average rating (weighted by review_count)
4. Median rating from actual user reviews
5. Top 5 cities by average rating
6. Bottom 5 cities
7. Consistency score = STDDEV of restaurant ratings
8. Year-over-year review volume trend for last 3 years

In [7]:
#Total Restaurants
df_combined['business_id'].drop_duplicates().count()



np.int64(67704)

In [8]:
duckdb.sql("select count(distinct business_id) as total_restaurants from filtered_df ")

┌───────────────────┐
│ total_restaurants │
│       int64       │
├───────────────────┤
│             67704 │
└───────────────────┘

In [9]:
df_category_aggregates = df_combined.groupby('categories').agg(
    total_restaurants=('business_id', 'nunique'),
    total_reviews=('review_id', 'count'),
    median_rating=('stars_y', 'median')
).reset_index()

df_category_aggregates

,categories,total_restaurants,total_reviews,median_rating
0,"Acai Bowls, American (New), Restaurants, Fast ...",1,153,4.0
1,"Acai Bowls, Asian Fusion, Food, Restaurants, B...",1,247,5.0
2,"Acai Bowls, Breakfast & Brunch, Coffee & Tea, ...",1,9,4.0
3,"Acai Bowls, Coffee & Tea, Food, Juice Bars & S...",1,48,4.5
4,"Acai Bowls, Coffee & Tea, Juice Bars & Smoothi...",2,14,5.0
...,...,...,...,...
39738,"Yoga, Cafes, Fitness & Instruction, Active Lif...",1,7,5.0
39739,"Yoga, Food, Active Life, Fitness & Instruction...",1,7,5.0
39740,"Yoga, Food, Cafes, Bakeries, Coffee & Tea, Act...",1,12,5.0
39741,"Ziplining, Team Building Activities, Kids Acti...",1,129,5.0


In [10]:
query = f"""
select
categories,
count(distinct business_id) as total_restaurants,
count(distinct review_id) as total_reviews,
median(stars_y) as median_rating
from df_combined
group by all
"""
duckdb.sql(query).df()

,categories,total_restaurants,total_reviews,median_rating
0,"Sushi Bars, Pubs, Gastropubs, Japanese, Restau...",1,318,4.0
1,"Juice Bars & Smoothies, Health Markets, Vegan,...",1,6,5.0
2,"Ice Cream & Frozen Yogurt, Food, Desserts, Sha...",2,13,4.0
3,"Restaurants, Asian Fusion, Japanese, Chinese",1,166,5.0
4,"Restaurants, Food Trucks, Mexican, Tex-Mex, Ta...",1,19,5.0
...,...,...,...,...
39738,"Food Trucks, Restaurants, American (Traditiona...",1,68,5.0
39739,"Distilleries, Medical Supplies, Food, Shopping...",1,5,5.0
39740,"Food, Restaurants, Breakfast & Brunch, Dessert...",1,53,5.0
39741,"Pool Halls, Nightlife, Dive Bars, Bars",1,8,4.0


In [11]:
import numpy as np

# 1. Create a temporary DataFrame with unique business-category pairs and their ratings/review counts
df_business_ratings_per_category = df_combined[['categories', 'business_id', 'stars_x', 'review_count']].drop_duplicates()

# 2. Group by 'categories' and calculate the weighted average rating and consistency score
df_category_metrics = df_business_ratings_per_category.groupby('categories').agg(
    weighted_avg_rating=pd.NamedAgg(column='stars_x', aggfunc=lambda x: np.average(x, weights=df_business_ratings_per_category.loc[x.index, 'review_count'])),
    consistency_score=pd.NamedAgg(column='stars_x', aggfunc='std')
).reset_index()

# Handle potential NaN values for consistency_score (e.g., categories with only one business)
df_category_metrics['consistency_score'] = df_category_metrics['consistency_score'].fillna(0)

# 3. Merge with df_category_aggregates
df_category_aggregates = pd.merge(df_category_aggregates, df_category_metrics, on='categories', how='left')

df_category_aggregates

,categories,total_restaurants,total_reviews,median_rating,weighted_avg_rating,consistency_score
0,"Acai Bowls, American (New), Restaurants, Fast ...",1,153,4.0,3.000000,0.000000
1,"Acai Bowls, Asian Fusion, Food, Restaurants, B...",1,247,5.0,4.000000,0.000000
2,"Acai Bowls, Breakfast & Brunch, Coffee & Tea, ...",1,9,4.0,4.000000,0.000000
3,"Acai Bowls, Coffee & Tea, Food, Juice Bars & S...",1,48,4.5,4.000000,0.000000
4,"Acai Bowls, Coffee & Tea, Juice Bars & Smoothi...",2,14,5.0,4.642857,0.707107
...,...,...,...,...,...,...
39738,"Yoga, Cafes, Fitness & Instruction, Active Lif...",1,7,5.0,5.000000,0.000000
39739,"Yoga, Food, Active Life, Fitness & Instruction...",1,7,5.0,5.000000,0.000000
39740,"Yoga, Food, Cafes, Bakeries, Coffee & Tea, Act...",1,12,5.0,4.500000,0.000000
39741,"Ziplining, Team Building Activities, Kids Acti...",1,129,5.0,5.000000,0.000000


In [12]:
# Calculate average business rating (stars_x) per city
city_avg_ratings = df_combined.groupby('city')['stars_x'].mean().reset_index()

# Get top 5 cities by average rating
top_5_cities = city_avg_ratings.sort_values(by='stars_x', ascending=False).head(5)

print("Top 5 cities by average rating:")
display(top_5_cities)

Top 5 cities by average rating:


,city,stars_x
1010,​Largo,5.0
940,West Norriton,5.0
986,Yardley Boro,5.0
872,Truckee,5.0
911,Wallingford,5.0


In [13]:
# Get bottom 5 cities by average rating
bottom_5_cities = city_avg_ratings.sort_values(by='stars_x', ascending=True).head(5)

print("Bottom 5 cities by average rating:")
display(bottom_5_cities)

Bottom 5 cities by average rating:


,city,stars_x
614,Peerless Park,1.0
48,Bellville,1.5
419,Land O'Lakes,1.5
970,Woodbury Hts.,1.5
491,Mc Cordsville,1.5


In [14]:
# Ensure 'date_x' is in datetime format
df_combined['date_x'] = pd.to_datetime(df_combined['date_x'])

# Extract the year from the review date
df_combined['review_year'] = df_combined['date_x'].dt.year


#YoY analysis
query = f"""
select
*,lag(total_reviews) over (order by review_year) as prev_year_reviews,
(total_reviews - lag(total_reviews) over (order by review_year))*100/lag(total_reviews) over (order by review_year) as yoy_change
from
(SELECT
    review_year,
    COUNT(DISTINCT review_id) AS total_reviews,
FROM
    df_combined
WHERE
    date_x >= (SELECT MAX(date_x) FROM df_combined) - INTERVAL 3 YEAR
GROUP BY ALL
order by review_year desc)
"""
duckdb.sql(query).df()


,review_year,total_reviews,prev_year_reviews,yoy_change
0,2019,640527,<NA>,NaN
1,2020,401206,640527,-37.363140
2,2021,457068,401206,13.923521
3,2022,23096,457068,-94.946923


# Q1.b — A/B Test Evaluation Using Real Yelp Data
We have two experiment groups:
1. Group A (Control): All cities except Las Vegas and Phoenix
2. Group B (Treatment): Only Las Vegas and Phoenix

Using the review, business, and check-in tables from the dataset:
Compute for each group, before vs after experiment date (use 2021-06-01 as split):
1. Average review length
2. Change in count of check-in’s
3. Change in % of 5-star reviews
4. A proxy for Sentiment Analysis using keyword counts
5. ATE (Average Treatment Effect)/AB Output on the following metrics:
a. Review_count
b. Average_stars
c. Check-ins
6. Run a Difference-in-Differences (DID) (pre-post A/B analysis) comparison in SQL

In [15]:
#making data ready for the a/b test analysis
experiment_date = '2021-06-01'
treatment_cities = ['Las Vegas', 'Phoenix']
experiment_date_dt = pd.to_datetime(experiment_date)

df_combined['group'] = df_combined['city'].apply(lambda x: 'Treatment' if x in treatment_cities else 'Control')
df_combined['period'] = df_combined['date_x'].apply(lambda x: 'After' if x >= experiment_date_dt else 'Before')

print("Added 'group' and 'period' columns to df_combined.")
df_combined[['city', 'date_x', 'group', 'period']]

Added 'group' and 'period' columns to df_combined.


,city,date_x,group,period
0,Philadelphia,2014-12-23 21:00:47,Control,Before
1,Philadelphia,2014-10-29 14:38:09,Control,Before
2,Philadelphia,2017-04-23 04:44:24,Control,Before
3,Philadelphia,2018-05-05 22:26:47,Control,Before
4,Philadelphia,2016-03-13 03:26:59,Control,Before
...,...,...,...,...
5243717,Edmonton,2017-12-19 03:58:37,Control,Before
5243718,Edmonton,2017-08-17 21:08:34,Control,Before
5243719,Edmonton,2018-07-25 04:17:01,Control,Before
5243720,Edmonton,2018-02-21 22:02:09,Control,Before


In [16]:
# 1. Feature Engineering for Metrics
# Average review length [cite: 32]
df_combined['review_length'] = df_combined['text'].str.len()

# 5-star review flag [cite: 34]
df_combined['is_5_star'] = (df_combined['stars_y'] == 5).astype(int)

# Proxy for Sentiment Analysis using keyword counts [cite: 35, 47]
# We count occurrences of positive keywords vs total words
pos_keywords = ['excellent', 'amazing', 'great', 'good', 'best', 'love']
df_combined['sentiment_proxy'] = df_combined['text'].str.lower().apply(
    lambda x: sum(x.count(key) for key in pos_keywords) if isinstance(x, str) else 0
)


In [17]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5243722 entries, 0 to 5243721
Data columns (total 29 columns):
 #   Column           Dtype         
---  ------           -----         
 0   business_id      object        
 1   name             object        
 2   address          object        
 3   city             object        
 4   state            object        
 5   postal_code      object        
 6   latitude         float64       
 7   longitude        float64       
 8   stars_x          float64       
 9   review_count     int64         
 10  is_open          int64         
 11  attributes       object        
 12  categories       object        
 13  hours            object        
 14  review_id        object        
 15  user_id          object        
 16  stars_y          float64       
 17  useful           int64         
 18  funny            int64         
 19  cool             int64         
 20  text             object        
 21  date_x           datetime64[us]

In [18]:
# Define sentiment keywords
pos_words = 'excellent|amazing|great|good|best|love|delicious'

query = f"""
WITH experimental_base AS (
    SELECT 
        *,
        CASE WHEN city IN ('Las Vegas', 'Phoenix') THEN 1 ELSE 0 END AS is_treatment,
        CASE WHEN CAST(date_x AS DATE) >= '2021-06-01' THEN 1 ELSE 0 END AS is_post,
        -- Metrics
        length(text) as review_length,
        CASE WHEN stars_y = 5 THEN 1 ELSE 0 END AS is_5_star,
        len(regexp_extract_all(lower(text), '({pos_words})')) AS pos_sentiment_count
    FROM df_combined
),
grouped_metrics AS (
    SELECT 
        is_treatment,
        is_post,
        -- Use COALESCE to ensure we don't have NULLs in our averages
        COALESCE(AVG(review_length), 0) as avg_len,
        COUNT(review_id) as total_reviews,
        COALESCE(AVG(is_5_star), 0) * 100 as pct_5_star,
        COALESCE(AVG(pos_sentiment_count), 0) as avg_sentiment,
        COALESCE(AVG(stars_y), 0) as avg_rating
    FROM experimental_base
    GROUP BY 1, 2
),

did_pivot AS (
    SELECT
        -- Volume/Check-in Proxy
        MAX(CASE WHEN is_treatment=1 AND is_post=1 THEN total_reviews ELSE 0 END) as T_Post_Vol,
        MAX(CASE WHEN is_treatment=1 AND is_post=0 THEN total_reviews ELSE 0 END) as T_Pre_Vol,
        MAX(CASE WHEN is_treatment=0 AND is_post=1 THEN total_reviews ELSE 0 END) as C_Post_Vol,
        MAX(CASE WHEN is_treatment=0 AND is_post=0 THEN total_reviews ELSE 0 END) as C_Pre_Vol,
        -- Ratings
        MAX(CASE WHEN is_treatment=1 AND is_post=1 THEN avg_rating ELSE 0 END) as T_Post_Stars,
        MAX(CASE WHEN is_treatment=1 AND is_post=0 THEN avg_rating ELSE 0 END) as T_Pre_Stars,
        MAX(CASE WHEN is_treatment=0 AND is_post=1 THEN avg_rating ELSE 0 END) as C_Post_Stars,
        MAX(CASE WHEN is_treatment=0 AND is_post=0 THEN avg_rating ELSE 0 END) as C_Pre_Stars,
        -- 5-Star %
        MAX(CASE WHEN is_treatment=1 AND is_post=1 THEN pct_5_star ELSE 0 END) as T_Post_5,
        MAX(CASE WHEN is_treatment=1 AND is_post=0 THEN pct_5_star ELSE 0 END) as T_Pre_5,
        MAX(CASE WHEN is_treatment=0 AND is_post=1 THEN pct_5_star ELSE 0 END) as C_Post_5,
        MAX(CASE WHEN is_treatment=0 AND is_post=0 THEN pct_5_star ELSE 0 END) as C_Pre_5,
        -- Review Length
        MAX(CASE WHEN is_treatment=1 AND is_post=1 THEN avg_len ELSE 0 END) as T_Post_Len,
        MAX(CASE WHEN is_treatment=1 AND is_post=0 THEN avg_len ELSE 0 END) as T_Pre_Len,
        MAX(CASE WHEN is_treatment=0 AND is_post=1 THEN avg_len ELSE 0 END) as C_Post_Len,
        MAX(CASE WHEN is_treatment=0 AND is_post=0 THEN avg_len ELSE 0 END) as C_Pre_Len
    FROM grouped_metrics
)
SELECT 'Review Count (Check-in Proxy) ATE' as Metric, (T_Post_Vol - T_Pre_Vol) - (C_Post_Vol - C_Pre_Vol) as ATE_Result FROM did_pivot
UNION ALL
SELECT 'Average Star Rating ATE', (T_Post_Stars - T_Pre_Stars) - (C_Post_Stars - C_Pre_Stars) FROM did_pivot
UNION ALL
SELECT '5-Star Percentage ATE', (T_Post_5 - T_Pre_5) - (C_Post_5 - C_Pre_5) FROM did_pivot
UNION ALL
SELECT 'Average Review Length ATE', (T_Post_Len - T_Pre_Len) - (C_Post_Len - C_Pre_Len) FROM did_pivot;
"""

duckdb.sql(query).df()

,Metric,ATE_Result
0,Review Count (Check-in Proxy) ATE,4.658526e+06
1,Average Star Rating ATE,4.117349e-03
2,5-Star Percentage ATE,-7.968223e+00
3,Average Review Length ATE,6.577696e+01


In [19]:
query = f"""
select
    "group",
    period,
    avg(review_length) as avg_review_length,
    count(distinct date_y) as avg_checkin_count,
    avg(is_5_star) as avg_pct_5_star,
    avg(sentiment_proxy) as avg_sent_proxy,
    avg(stars_y) as avg_stars,
    count(distinct review_id) as avg_review_count
from df_combined
group by all
"""
duckdb.sql(query).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,group,period,avg_review_length,avg_checkin_count,avg_pct_5_star,avg_sent_proxy,avg_stars,avg_review_count
0,Control,After,481.056610,35972,0.520284,1.527539,3.797435,292598
1,Control,Before,546.833572,65387,0.440602,1.701573,3.801553,4951124


# Q1.c — Restaurant Operational Risk Detection
Using review + business tables:
Identify restaurants with signals of operational issues using review patterns.
For each restaurant compute:
1. % of reviews containing negative keywords: “slow”, “cold”, “rude”, “late”, “wait”, “service”
2. Rolling 3-month average rating
3. Check if restaurant’s rating has fallen by > 0.8 over any 6 month window
4. Spike in negative keywords
5. Standard deviation in Ratings
6. Define a formula for “Operational Risk Score” (you can assume what goes into the
formula) - This metric should be a composite score of all metrics that make it risky to
continue operations for this restaurant
Return the top 50 highest-risk restaurants per major city.

In [20]:
query = """
WITH base_data AS (
    SELECT
        business_id,
        name,
        city,
        stars_x AS review_stars,
        CAST(date_y AS DATE) as review_date,
        -- Check for negative keywords
        CASE WHEN
            regexp_matches(lower(text), 'slow|cold|rude|late|wait|service')
            THEN 1 ELSE 0 END AS is_neg_keyword
    from df_combined
),
rolling_metrics AS (
    SELECT
        *,
        -- 2. Rolling 3-month average
        AVG(review_stars) OVER (
            PARTITION BY business_id ORDER BY review_date
            RANGE BETWEEN INTERVAL '3' MONTH PRECEDING AND CURRENT ROW
        ) as rolling_3m_avg,
        -- 3. Check for 6-month drop logic (Max in last 6 months - current rating)
        MAX(review_stars) OVER (
            PARTITION BY business_id ORDER BY review_date
            RANGE BETWEEN INTERVAL '6' MONTH PRECEDING AND CURRENT ROW
        ) as rolling_6m_max
    FROM base_data
),
restaurant_summary AS (
    SELECT
        business_id, name, city,
        -- 1. % Negative Keywords
        AVG(is_neg_keyword) as pct_neg_reviews,
        -- 5. Std Deviation
        STDDEV(review_stars) as rating_std,
        -- 3. Rating Drop > 0.8
        MAX(CASE WHEN (rolling_6m_max - review_stars) > 0.8 THEN 1 ELSE 0 END) as rating_drop_flag,
        COUNT(*) as total_reviews,
        AVG(review_stars) as lifetime_avg
    FROM rolling_metrics
    GROUP BY 1, 2, 3
),
final_scoring AS (
    SELECT *,
        -- 6. Operational Risk Score Formula
        (pct_neg_reviews * 40) +
        (rating_drop_flag * 30) +
        (COALESCE(rating_std, 0) * 20) +
        ((5 - lifetime_avg) * 10) as operational_risk_score
    FROM restaurant_summary
    WHERE total_reviews > 5 -- Statistical significance
),
ranked_risk AS (
    SELECT *,
        RANK() OVER (PARTITION BY city ORDER BY operational_risk_score DESC) as risk_rank
    FROM final_scoring
)
SELECT * FROM ranked_risk WHERE risk_rank <= 50;
"""

duckdb.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,business_id,name,city,pct_neg_reviews,rating_std,rating_drop_flag,total_reviews,lifetime_avg,operational_risk_score,risk_rank
0,yQh7_mOLWV2XlkbhTAk5CA,IHOP,BRANDON,0.814815,0.0,0,54,2.5,57.592593,1
1,tkyGbszNAP2w0VwNTi48qQ,Rita's Italian Ice,Bellefonte,0.444444,0.0,0,9,4.0,27.777778,1
2,tpSeNezanZDvsXWR7Hhi1w,Taco Bell,Carmel,0.666667,0.0,0,27,1.0,66.666667,1
3,S_tu8JCeZsbpgJsqRFaSBg,KFC,Carmel,0.725000,0.0,0,40,1.5,64.000000,2
4,MMdVzFD8vZcq-ZhblnqMXw,Wendy's,Carmel,0.716981,0.0,0,53,1.5,63.679245,3
...,...,...,...,...,...,...,...,...,...,...
16171,ZItR6OeBp4MX2LjqCS7TFw,"Uncle Dave's Homemade Ice Cream, LLC",Yardley,0.333333,0.0,0,18,4.0,23.333333,47
16172,4bS9BqDL0zUNxS8EJKYROw,Cairo Cakes,Yardley,0.452381,0.0,0,42,4.5,23.095238,48
16173,H4lnb4UI5V2DiMVp1NTytg,La La Lobster,Yardley,0.418182,0.0,0,55,4.5,21.727273,49
16174,xDFWKDY0sGyJYSMYc0uXWQ,Eliyahu's Yardley Pizza,Yardley,0.287356,0.0,0,87,4.0,21.494253,50


# Q1.d — Menu Competitiveness Index (Inferred From Reviews)
Using review text (no menu table available) from review table:
1. Build a Menu Variety Index using distinct food keywords mentioned
2. Build a Popularity Index: review_count × average_stars
3. Build a Price Index from business.attributes.PriceRange
4. Compute a Demand Concentration Score: high = few dishes repeatedly mentioned
5. Combine into a Competitiveness Score

Return the top 20 restaurants per city.


In [21]:
import duckdb

con = duckdb.connect(database=':memory:')
con.register('df_combined', df_combined)

# Standardized food keyword list
foods = ['burger', 'pizza', 'sushi', 'taco', 'pasta', 'salad', 'steak', 'chicken', 'fries', 'dessert']

# Boolean flags: check if a review contains a specific food
variety_flags = ", ".join([f"(lower(text) LIKE '%{f}%')::int AS has_{f}" for f in foods])

# Aggregation logic
variety_sum = " + ".join([f"MAX(has_{f})" for f in foods]) # Distinct keywords per business
total_mentions = " + ".join([f"SUM(has_{f})" for f in foods]) # Volume of mentions
max_mention = "GREATEST(" + ", ".join([f"SUM(has_{f})" for f in foods]) + ")" # Lead dish mentions

query = f"""
WITH base_flags AS (
    SELECT 
        business_id, name, city, stars_x, review_count, attributes,
        {variety_flags}
    FROM df_combined
),
restaurant_stats AS (
    SELECT 
        business_id, name, city, stars_x, review_count, attributes,
        ({variety_sum}) AS menu_variety_index,
        ({total_mentions}) AS total_score,
        ({max_mention}) AS max_score
    FROM base_flags
    GROUP BY 1, 2, 3, 4, 5, 6
),
final_calculation AS (
    SELECT 
        *,
        -- 2. Popularity Index
        (review_count * stars_x) AS popularity_index,
        
        -- 3. Price Index (Fixed for 'None' and string errors)
        COALESCE(TRY_CAST(json_extract_path(attributes, '$.RestaurantsPriceRange2') AS INTEGER), 2) AS price_index,
        
        -- 4. Demand Concentration Score (High = specialized/few dishes)
        CAST(max_score AS FLOAT) / NULLIF(total_score, 0) AS demand_concentration_score
    FROM restaurant_stats
    WHERE total_score > 0
)
SELECT * FROM (
    SELECT *, 
        -- 5. Final Competitiveness Score
        ((menu_variety_index * 0.3) + 
         (popularity_index * 0.3) + 
         ((5 - price_index) * 0.2) + 
         ((1 - demand_concentration_score) * 0.2)) AS competitiveness_score,
        RANK() OVER(PARTITION BY city ORDER BY competitiveness_score DESC) as rank
    FROM final_calculation
) WHERE rank <= 20;
"""

df_q1d_final = con.execute(query).df()
print(df_q1d_final)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 business_id                                 name  \
0     Qu3q0r-C4vC6yfVcetptLg            Cody's Original Roadhouse   
1     94jR79bbW47Dbl6XlNr3tw                           Le Bouchon   
2     K4ogLNLcwYNnwulNW8ikSg                 E & E Stakeout Grill   
3     3EUsW5xcLI4qoVWyfEzYVg           Maggie Mae's on the Bluffs   
4     RsZFGqDyTUk9kEJzVLWHdQ         Marlin Darlin Key West Grill   
...                      ...                                  ...   
8698  2w9N02Lv5mCZzHcioRlQ_A                        Goodson Farms   
8699  AneGQX4oAXH5scien9XHvA                               Publix   
8700  2y_CdkxEOJEJGyJApfCYpA  Rode's Fireside Restaurant & Tavern   
8701  ZngsxFOi6nobpDSjPigJ9A                Wycombe Publick House   
8702  gP5wF34lgcfI1yAaS4Vu4A                                Bacio   

                 city  stars_x  review_count  \
0        Belleair Blf      2.5           160   
1        Belleair Blf      3.5            15   
2     Belleair Bluffs      

In [22]:
df_q1d_final

,business_id,name,city,stars_x,review_count,attributes,menu_variety_index,total_score,max_score,popularity_index,price_index,demand_concentration_score,competitiveness_score,rank
0,Qu3q0r-C4vC6yfVcetptLg,Cody's Original Roadhouse,Belleair Blf,2.5,160,"{'RestaurantsAttire': 'u'casual'', 'Restaurant...",9,157.0,46.0,400.0,2,0.292994,123.441401,1
1,94jR79bbW47Dbl6XlNr3tw,Le Bouchon,Belleair Blf,3.5,15,"{'BusinessParking': '{'garage': False, 'street...",4,7.0,3.0,52.5,2,0.428571,17.664286,2
2,K4ogLNLcwYNnwulNW8ikSg,E & E Stakeout Grill,Belleair Bluffs,4.0,298,"{'Caters': 'True', 'HasTV': 'True', 'Restauran...",8,285.0,119.0,1192.0,2,0.417544,360.716491,1
3,3EUsW5xcLI4qoVWyfEzYVg,Maggie Mae's on the Bluffs,Belleair Bluffs,4.0,246,"{'RestaurantsTakeOut': 'True', 'RestaurantsDel...",8,52.0,26.0,984.0,1,0.500000,298.500000,2
4,RsZFGqDyTUk9kEJzVLWHdQ,Marlin Darlin Key West Grill,Belleair Bluffs,3.5,207,"{'OutdoorSeating': 'True', 'RestaurantsDeliver...",9,146.0,45.0,724.5,2,0.308219,220.788356,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8698,2w9N02Lv5mCZzHcioRlQ_A,Goodson Farms,Wimauma,5.0,6,"{'RestaurantsTakeOut': 'True', 'BikeParking': ...",1,1.0,1.0,30.0,2,1.000000,9.900000,5
8699,AneGQX4oAXH5scien9XHvA,Publix,Wimauma,3.5,6,"{'BusinessParking': '{'garage': False, 'street...",2,2.0,1.0,21.0,2,0.500000,7.600000,6
8700,2y_CdkxEOJEJGyJApfCYpA,Rode's Fireside Restaurant & Tavern,Woolwich Twp,3.5,104,"{'BusinessParking': '{'garage': False, 'street...",8,93.0,26.0,364.0,2,0.279570,112.344086,1
8701,ZngsxFOi6nobpDSjPigJ9A,Wycombe Publick House,Wycombe,3.5,79,"{'WiFi': 'u'free'', 'RestaurantsPriceRange2': ...",9,73.0,25.0,276.5,2,0.342466,86.381507,1
